### Install new libraries

In [43]:
#!pip install ddgs trafilatura

### Setup

In [44]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from pprint import pprint
from IPython.display import display, Markdown
from ddgs import DDGS
import trafilatura


# ------------------------------
# Setup
# ------------------------------

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
	raise Exception("API key is missing")
client = OpenAI()

MODEL = "gpt-4.1-mini"


### Step 1: Define the Tools

In [45]:
def search_web(query: str):
	"""Search the web using DuckDuckGo browser. Returns 3 results."""
	ddgs = DDGS()
	results = ddgs.text(query, max_results=3)
	print(f"  \u2705 Got results")
	return json.dumps(results, indent=2)

In [46]:
def fetch_url(url: str):
	"""Fetch the content of a URL using trafilatura."""
	downloaded = trafilatura.fetch_url(url)
	if downloaded:
		text =  trafilatura.extract(downloaded)
		if text:
			print(f"  \u2705 Got text: {len(text)} characters")
			return text
	print(f"  \u274C Failed to fetch or extract text.") # For debugging
	return f"Could not extract text from {url}. Try a different source." # For LLM input feedback


In [47]:
#Test search_web function with a sample query
#search_web("Apples in Madagascar") #Calling the search_web function to test it with a sample query(str)

#search_web("AI in healthcare in 2030") #Calling the search_web function to test it with a sample query(str)

In [48]:
#Test fetch_url function with a sample URL
#result = fetch_url("https://explurt.com/blog-english/madagascar-english/the-list-of-fruits-to-eat-in-madagascar/") #Calling the fetch_url function to test it with a sample URL(str)
#print(result)

### Step 2: Describe as LLM tools

In [49]:
tools = []

In [50]:
#Describe search_web function as an LLM tool
search_web_function = {
	"name": "search_web",
	"description": "Search the web using DuckDuckGo browser. Returns 3 results.",
	"parameters": {
		"type": "object",
		"properties": {
			"query": {
				"type": "string",
				"description": "The search query to find relevant websites."
			}
		},
		"required": ["query"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": search_web_function})

In [51]:
#Describe fetch_url function as an LLM tool
fetch_url_function = {
	"name": "fetch_url",
	"description": "Fetch and extract the main content from a web page.",
	"parameters": {
		"type": "object",
		"properties": {
			"url": {
				"type": "string",
				"description": "The URL of the web page to fetch and extract text from."
			}
		},
		"required": ["url"]
	}
}

#add to list of tools
tools.append({"type": "function", "function": fetch_url_function})


In [52]:
#Check the tools list
#tools

### Step 3: Tool Call handler

In [53]:
def handle_tool_call(tool_calls):
	tool_results = []


	for tool_call in tool_calls:  # handle every tool call the model requested
		function_name = tool_call.function.name
		args = json.loads(tool_call.function.arguments)  # arguments are JSON
		print(f"  \u0001F527 Calling function {function_name} with arguments {args}") # For future debugging

		#Route to the appropriate function based on function_name
		if function_name == "search_web":
			result = search_web(args["query"])
			content = f"Search results: {result}"

		elif function_name == "fetch_url":
			result = fetch_url(args["url"])
			content = f"Fetched URL content: {result}"
		else:
			content = f"Unknown function: {function_name}"
		tool_call_result = {
			"role": "tool",
			"content": content,
			"tool_call_id": tool_call.id
		}
		tool_results.append(tool_call_result)
	return tool_results

### Step 4: The System Prompt

 This tells the LLM who it is and how to behave. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce




In [54]:

# RESEARCH_AGENT_PROMPT = """You are a research agent that can search the web and fetch content from URLs.
# You have access to the following tools:
# 1. search_web(query: str) -> str: Search the web using DuckDuckGo browser. Returns 3 results.
# 2. fetch_url(url: str) -> str: Fetch and extract the main content from a web page.
# Your task is to answer questions and provide information by using these tools when necessary.\
# If you need to search for information, use the search_web tool. If you find a relevant URL, use the fetch_url tool to extract the content.\
# Always provide accurate and concise information in your responses. If you cannot find the information, respond with 'I could not find the information you requested.\
# Do not make up information. Always cite your sources when providing information from the web.\
# If you use the fetch_url tool, provide a summary of the content you extracted."""

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""


### Step 5: The Agentic Loop

In [55]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
	"""
	Run the research agent on a topic and return the research brief.
	Args:
		topic: The topic to research
		max_iterations: Safety limit to prevent infinite loops
	Returns:
		The research brief as a string
	"""
	print(f"\n🔍 Starting research on: {topic}")
	print("=" * 60)

	# Initialize conversation messages list with system prompt + research task
	messages = [
		{"role": "system", "content": RESEARCH_AGENT_PROMPT},
		{"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
	]

	# LOOP
	iteration = 0

	while iteration < max_iterations:
		iteration += 1
		print(f"\n---Iteration {iteration} ---")

		# 1. Call the LLM and get response
		response = client.chat.completions.create(
			model=MODEL,
			messages=messages,
			tools=tools,
		)

		message = response.choices[0].message
		messages.append(message)

		# 2. Check if LLM called tools
		if message.tool_calls:
			tool_results = handle_tool_call(message.tool_calls)
			messages.extend(tool_results)

		# 3. Otherwise: no tools were called, read message content
		else:
			content = message.content # No tool calls, so there's just a normal text response from the LLM
			# Check if DONE:, then return
			if content and content.startswith("DONE:"):
				research_brief = content[len("DONE:"):].strip()  # Remove "DONE:" prefix and whitespace
				print(f"  ✅ Research complete!")
				return research_brief

			# Otherwise: not yet done
			else:
				print(f"  💭 Agent is thinking...")
				pprint(content)
				# Loop continues to next iteration

		# 4. If we're entering the final iteration, force a final answer
		if iteration == max_iterations - 1:
			print("⚠️ Safety limit reached. Stopping research in next iteration.")
			messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your final research brief now, even if it is incomplete."})

	# Fallback return
	return "Research incomplete. Maximum iterations reached without finalizing brief."

### Let's Run IT!

In [56]:
brief = run_research_agent("The impact of AI on healthcare in 2030")
display(Markdown(f"### Research Brief on 'The impact of AI on healthcare in 2030'\n\n{brief}"))


🔍 Starting research on: The impact of AI on healthcare in 2030

---Iteration 1 ---
  F527 Calling function search_web with arguments {'query': 'impact of AI on healthcare in 2030'}
  ✅ Got results

---Iteration 2 ---
  F527 Calling function fetch_url with arguments {'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12764347/'}
  ❌ Failed to fetch or extract text.
  F527 Calling function fetch_url with arguments {'url': 'https://ep.jhu.edu/news/ai-in-healthcare-applications-and-impact/'}
  ❌ Failed to fetch or extract text.
  F527 Calling function fetch_url with arguments {'url': 'https://healthmanagement.org/index.php/s/stanford-how-artificial-intelligence-will-transform-healthcare-by-2030'}
  ✅ Got text: 2201 characters

---Iteration 3 ---
  F527 Calling function search_web with arguments {'query': 'impact of AI on healthcare market growth by 2030'}
  ✅ Got results

---Iteration 4 ---
  F527 Calling function fetch_url with arguments {'url': 'https://finance.yahoo.com/news/ai-hea

### Research Brief on 'The impact of AI on healthcare in 2030'

Research Brief: The Impact of AI on Healthcare in 2030

Key Facts and Statistics:
- The global AI in healthcare market is projected to grow from USD 26.6 billion in 2024 to approximately USD 187.7 billion by 2030, with a compound annual growth rate (CAGR) of around 38.5% (PR Newswire, The Research Insights).
- The US AI healthcare market alone is expected to grow from about $8.65 billion in 2025 to over $43 billion by 2030, representing a 38% CAGR (MarketsandMarkets).
- North America commands over 45% of the global AI healthcare market share, with rapid growth observed also in Europe and Asia-Pacific driven by government initiatives and technological infrastructure.

Main Themes and Arguments:
1. Market Expansion & Key Technologies:
   - AI adoption in healthcare will span clinical decision support, robotic surgeries, patient monitoring, imaging diagnostics, drug discovery, and administrative automation.
   - Machine learning, natural language processing, and computer vision will power these AI applications.
   - Robotic-assisted surgery leads market revenue among AI applications and is expected to continue growing strongly through 2030.

2. Improved Healthcare Outcomes and Efficiency:
   - Early disease detection via AI-driven pattern recognition in imaging reduces mortality and treatment costs.
   - Personalized medicine will be enabled by AI's integration of genomics, patient history, and lifestyle data, aligning with value-based healthcare models.
   - Virtual assistants and remote monitoring devices powered by AI will enhance chronic disease management and elder care.
   - AI will help mitigate workforce shortages by automating administrative and some cognitive clinical tasks.

3. Challenges and Adoption Factors:
   - Widespread implementation depends on overcoming regulatory hurdles, ensuring data privacy/security, and building clinicians' and public trust.
   - Trustworthy, transparent AI systems that respect human rights and deliver explainable results are critical for adoption.
   - Data governance, regulatory clarity, and user-friendly interfaces are essential for AI’s future integration.

4. Regional Insights:
   - North America leads due to strong healthcare IT infrastructure, investment, and innovation ecosystems.
   - Europe’s growth is boosted by public-private partnerships and dedicated funding for AI-healthcare projects.
   - Asia-Pacific benefits from a booming startup ecosystem, rising healthcare investments, and government programs supporting AI technology use.

Notable Data Points:
- AI in Oncology market is forecast to grow at nearly 29% CAGR, reaching substantial market value by 2030.
- The Generative AI healthcare market is projected to reach nearly $14.8 billion by 2030.
- AI-powered administrative workflow assistants and virtual health assistants significantly reduce operational costs.
- AI-driven drug discovery accelerates R&D timelines and increases precision, reducing costs and improving outcomes.

Sources:
- PR Newswire via The Research Insights: https://finance.yahoo.com/news/ai-healthcare-market-revenue-worth-152500085.html
- MarketsandMarkets US AI Healthcare Market report: https://www.marketsandmarkets.com/Market-Reports/geography/artificial-intelligence-healthcare-market/us
- Stanford research summary on healthcare AI transformation by 2030: https://healthmanagement.org/index.php/s/stanford-how-artificial-intelligence-will-transform-healthcare-by-2030

Summary:
By 2030, AI is poised to revolutionize healthcare through massive market growth and technological innovation. Key applications such as robotic surgery, diagnostics, personalized treatment plans, and remote monitoring will drive better health outcomes and system efficiencies. Challenges remain in regulation and trust, but strong investment, research, and adoption trends predict AI as a foundational technology in future healthcare delivery globally.